# Week 2: Clean the Online Retail data

**Your job:** decide which invoice lines belong in a *positive-sales* analysis, document the decision, and create a cleaned file. You write the code in the **Your code** cells. Run cells **one at a time** with **Shift+Enter**. Double-click a Markdown cell to edit your written answers, then save with **⌘S**.

A row is an **invoice line item**; one invoice can have several rows. We will count distinct invoices when we want an order count. A positive-sales total is **not** a full accounting of net revenue, because returns and cancellations are set aside for separate analysis.

The original workbook stays unchanged in `data/raw/`. Your new file will go in `data/processed/`.

## 1. Load the original workbook

This cell finds the project folder, reads the Excel file, and treats IDs as text. Run it first. Loading the full workbook may take a little while.

In [5]:
from pathlib import Path
import pandas as pd

project_root = next(
    (folder for folder in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (folder / "data" / "raw" / "Online Retail.xlsx").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Put Online Retail.xlsx in data/raw/ and reopen this notebook.")

raw = pd.read_excel(
    project_root / "data" / "raw" / "Online Retail.xlsx",
    engine="openpyxl",
    dtype={"InvoiceNo": "string", "StockCode": "string", "CustomerID": "string"},
)
print(f"Loaded {len(raw):,} rows and {raw.shape[1]} columns")
display(raw.head())

Loaded 541,909 rows and 8 columns


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


## 2. Count the quality issues before cleaning

Create these four variables and print their values:

- `cancelled_count`: invoice numbers starting with `"C"`
- `nonpositive_quantity_count`: `Quantity` at or below zero
- `nonpositive_price_count`: `UnitPrice` at or below zero
- `missing_customer_count`: missing `CustomerID`

**Hints:** `.str.startswith("C", na=False)` checks invoice IDs; `<= 0` checks nonpositive numbers; `.isna()` checks missing values; `.sum()` counts `True` results. Keep these counts separate: a row can satisfy more than one condition.

In [20]:
# Your code: calculate and print the four counts.
cancelled_count = len(raw[(raw["InvoiceNo"].str.startswith("C", na = False))])
nonpositive_quantity_count = len(raw[(raw["Quantity"] <= 0)])
nonpositive_price_count = len(raw[(raw["UnitPrice"] <= 0)])
missing_customer_count = len(raw[raw["CustomerID"].isna()])

display(cancelled_count)
display(nonpositive_quantity_count)
display(nonpositive_price_count)
display(missing_customer_count)



9288

10624

2517

135080

**My counts:**

- Cancelled invoice lines: 9288
- Nonpositive quantities: 10624
- Nonpositive prices: 2517
- Missing customer IDs: 135080

**My interpretation:** Which of these issues can affect a sales total? Which mainly affects customer-level questions?

I think that the first 3 affect revenue calculation the most because it directly affects how many actually converted into sales. However the invoices with missing customer IDs will affect customer related questions.


## 3. Build masks for the cleaning rule

For this project, a *positive-sales line* must satisfy all three rules:

1. Its `InvoiceNo` does **not** begin with `C`.
2. Its `Quantity` is greater than zero.
3. Its `UnitPrice` is greater than zero.

Create Boolean masks named `cancelled`, `bad_quantity`, and `bad_price`. Then combine them into `excluded` with `|` (OR). This marks each row to leave out **once**, even when it breaks several rules. Print `excluded.sum()` and compare it with the four counts above. Do **not** exclude a row solely because `CustomerID` is missing.

In [60]:
# Your code: create cancelled, bad_quantity, bad_price, and excluded.
# Hint: use parentheses around each comparison when combining conditions.
cancelled = raw["InvoiceNo"].str.startswith("C", na=False)
bad_quantity = raw["Quantity"] <= 0
bad_price = raw["UnitPrice"] <= 0
customer_empty = raw["CustomerID"].isna()

excluded = (cancelled | bad_quantity | bad_price)

display(raw[~excluded & customer_empty])

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,2010-12-01 14:32:00,2.51,<NA>,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,2010-12-01 14:32:00,2.51,<NA>,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,2010-12-01 14:32:00,0.85,<NA>,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,2010-12-01 14:32:00,1.66,<NA>,United Kingdom
1447,536544,21790,VINTAGE SNAP CARDS,9,2010-12-01 14:32:00,1.66,<NA>,United Kingdom
...,...,...,...,...,...,...,...,...
541536,581498,85099B,JUMBO BAG RED RETROSPOT,5,2011-12-09 10:26:00,4.13,<NA>,United Kingdom
541537,581498,85099C,JUMBO BAG BAROQUE BLACK WHITE,4,2011-12-09 10:26:00,4.13,<NA>,United Kingdom
541538,581498,85150,LADIES & GENTLEMEN METAL SIGN,1,2011-12-09 10:26:00,4.96,<NA>,United Kingdom
541539,581498,85174,S/4 CACTI CANDLES,1,2011-12-09 10:26:00,10.79,<NA>,United Kingdom


**My check:** The total excluded rows is 11805. Why might this differ from adding the three individual issue counts?

It might differ because some rows might qualify for multiple conditions at once, leading to double counting of certain invoices.

## 4. Keep the positive-sales lines

Use your `excluded` mask to make a new DataFrame named `sales`. `.loc[...]` selects rows; `~excluded` means "not excluded"; `.copy()` gives you a separate working table. Leave `raw` unchanged.

After you write the code, run the check cell below. If it fails, use its message to revisit your filter.

In [36]:
# Your code: create sales from raw using your excluded mask.
sales = raw.loc[~excluded].copy()
print(len(sales))
display(sales)

530104


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France


In [37]:
# Check your filter. This cell gives feedback; it does not change the data.
assert "sales" in globals(), "Create a DataFrame named sales in the cell above."
assert len(sales) == len(raw) - int(excluded.sum()), "Check how you applied excluded."
assert not sales["InvoiceNo"].str.startswith("C", na=False).any(), "A cancelled invoice remains."
assert (sales["Quantity"] > 0).all(), "A nonpositive quantity remains."
assert (sales["UnitPrice"] > 0).all(), "A nonpositive price remains."
print(f"Checks passed: kept {len(sales):,} of {len(raw):,} rows.")

Checks passed: kept 530,104 of 541,909 rows.


**My decision:** Explain in one or two sentences why you kept or excluded these rows. What is one limitation of calling the resulting total "sales"?

## 5. Calculate line revenue

Each retained line contributes `Quantity × UnitPrice` pounds. Create a `Revenue` column in `sales`. Use the exact column name `Revenue` so the check cell works. Then calculate its total. This is recorded positive sales, **not profit**; the data does not include costs.

In [ ]:
# Your code: create sales["Revenue"] and print total positive-sales revenue.
sales["Revenue"] = sales["Quantity"] * sales["UnitPrice"]
display(sales["Revenue"])


np.float64(10666684.544)

In [42]:
# Check the revenue column.
assert "Revenue" in sales.columns, "Add a Revenue column in the cell above."
assert sales["Revenue"].notna().all(), "Revenue has missing values."
assert (sales["Revenue"] > 0).all(), "Revenue should be positive in this table."
print("Revenue column passed the checks.")

Revenue column passed the checks.


## 6. Prepare the date for later trend analysis

Convert `InvoiceDate` to a datetime with `pd.to_datetime(...)`. Then make a `YearMonth` column such as `2011-03` using `.dt.strftime("%Y-%m")`. Show the first few dates and months to check your result.

In [53]:
# Your code: convert InvoiceDate and create YearMonth.
pd.to_datetime(sales["InvoiceDate"])
sales.head()

sales["YearMonth"] = sales["InvoiceDate"].dt.strftime("%Y-%m")
sales.head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,YearMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12


In [54]:
# Check the month field.
assert "YearMonth" in sales.columns, "Add the YearMonth column above."
assert sales["YearMonth"].notna().all(), "Some months are missing."
print("Month field passed the checks.")

Month field passed the checks.


## 7. Save a processed CSV

Save `sales` to `data/processed/valid_sales.csv`. Build the path from `project_root`, then use `.to_csv(..., index=False)`. The `index=False` option avoids adding a numbered column. Keep the original workbook in `data/raw/`.

In [55]:
# Your code: save the processed CSV and print its path.
output_path = project_root / "data" / "processed" / "valid_sales.csv"
sales.to_csv(output_path, index=False)
print(output_path)
print(output_path.is_file())


/Users/abepark/Documents/Codex/2026-10-28/referenced-chatgpt-conversation-this-is-an/ecommerce-sales-analysis/data/processed/valid_sales.csv
True


## 8. Record your cleaning log

Edit this cell using your actual results. **Do not** add the separate exclusion counts to get the total removed: the rules can overlap.

- Original invoice lines: 540K~
- Lines marked for exclusion by at least one rule: 11K~
- Positive-sales lines kept: 530K~
- Positive-sales lines with missing customer IDs: 130K~
- Total recorded positive-sales revenue: £10mil~

**Rules used:** Can't have negative quantity, cant be cancelled, cant have negative price

**Why missing customer IDs were kept or excluded:** Possibly because you can't link it to a particular customer.

**What this positive-sales total does not measure:** Does not measure profit.

**One question for Week 3:** not sure.

When your code runs and this log is filled in, we can move the repeatable cleaning steps into `src/clean_data.py` and publish Week 2 to GitHub.